# 02 · Lista de casa: pandas 3

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- **Não percorra linhas**: nada de `for` sobre a tabela, `iterrows` ou `apply(axis=1)`.
- O notebook exige pandas 3 (a primeira célula confere).
- As células ✅ conferem formato e alguns valores; passar nelas não garante nota máxima.
- Antes de entregar, use **Reiniciar sessão e executar tudo**.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Auditoria de uma tabela | 1,5 |
| 2 | Código legado e *Copy-on-Write* | 2,0 |
| 3 | Subcategorias com `groupby` | 2,0 |
| 4 | Clientes: agregação e junção | 2,0 |
| 5 | Evolução semanal | 1,5 |
| 6 | Três achados, por escrito | 1,0 |

In [ ]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

assert int(pd.__version__.split(".")[0]) >= 3, "Atualize: %pip install -q -U 'pandas>=3,<4' e reinicie a sessão."

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, low_memory=False, dtype={"product_id": str}, parse_dates=["submission_date"])
print(df.shape)

## Questão 1 (1,5 ponto): auditoria de uma tabela

Escreva `auditar(tabela)`, que devolve um `DataFrame` com **uma linha por coluna** de `tabela` (o nome da coluna no índice) e as colunas:

- `tipo`: o tipo da coluna, como texto;
- `ausentes`: a proporção de valores ausentes;
- `distintos`: o número de valores distintos;
- `exemplo`: o primeiro valor não ausente da coluna.

O resultado deve vir ordenado da maior para a menor proporção de ausentes.

In [ ]:
# ✏️ Questão 1
# SEU CÓDIGO AQUI
def auditar(tabela):
    ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
_a = auditar(df)
assert list(_a.columns) == ["tipo", "ausentes", "distintos", "exemplo"] and len(_a) == df.shape[1]
assert _a.index[0] == "product_brand" and _a["ausentes"].is_monotonic_decreasing
assert _a.loc["overall_rating", "distintos"] == 5 and _a.loc["review_text", "tipo"] == "str"
_a

**1b)** Com base na auditoria, cite uma coluna que você **não** usaria em um modelo e uma coluna que exigiria limpeza antes do uso. Justifique cada escolha em uma frase.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,0 pontos): código legado e *Copy-on-Write*

A função abaixo foi escrita para o pandas 1. Ela roda no pandas 3 sem erro e **não faz** parte do que promete.

In [ ]:
def limpar_legado(tabela):
    """Deveria: (1) trocar anos de nascimento impossíveis por ausente; (2) preencher gênero ausente com 'N';
    (3) padronizar o estado em maiúsculas; (4) criar a coluna `idade`."""
    tabela[tabela["reviewer_birth_year"] < 1918]["reviewer_birth_year"] = np.nan
    tabela["reviewer_gender"].fillna("N", inplace=True)
    estados = tabela["reviewer_state"]
    estados = estados.str.upper()
    tabela["idade"] = 2018 - tabela["reviewer_birth_year"]
    return tabela

**2a)** Preencha `funciona` com `True` ou `False` para cada uma das quatro tarefas, indicando se ela é de fato realizada no pandas 3. Responda antes de executar; depois confira.

**2b)** Escreva `limpar(tabela)`, que realiza as quatro tarefas, devolve uma tabela nova e não gera nenhum aviso. Considere impossíveis os anos anteriores a 1918 **e** os posteriores a 2004.

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
funciona = {1: ..., 2: ..., 3: ..., 4: ...}

def limpar(tabela):
    ...

In [ ]:
# ✅ Conferência da questão 2 (não altere)
_copia = df.copy()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _leg = limpar_legado(df.copy())
_real = {
    1: bool((_leg["reviewer_birth_year"] < 1918).sum() == 0),
    2: bool(_leg["reviewer_gender"].isna().sum() == 0),
    3: False,
    4: "idade" in _leg.columns,
}
print(f"Previsões corretas em 2a: {sum(funciona.get(k) == v for k, v in _real.items())} de 4")
with warnings.catch_warnings():
    warnings.simplefilter("error")
    _l = limpar(df)
assert df.equals(_copia), "a tabela recebida foi alterada"
assert _l["reviewer_birth_year"].min() >= 1918 and _l["reviewer_birth_year"].max() <= 2004
assert _l["reviewer_gender"].isna().sum() == 0 and set(_l["reviewer_gender"].unique()) == {"F", "M", "N"}
assert _l["idade"].min() >= 14 and _l["idade"].max() <= 100
print("Formato correto.")

**2c)** Para cada tarefa que a função legada não realiza, explique o motivo em uma frase, usando a regra do *Copy-on-Write*.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (2,0 pontos): subcategorias

Construa `subcategorias`, com uma linha por `site_category_lv2` e as colunas:

- `avaliacoes`: número de avaliações;
- `nota_media`;
- `pct_nota_1`: proporção de avaliações com nota 1;
- `diferenca`: `pct_nota_1` da subcategoria menos a proporção de nota 1 **na tabela inteira**.

Mantenha as subcategorias com pelo menos 800 avaliações e ordene da maior para a menor `diferenca`.

In [ ]:
# ✏️ Questão 3
# SEU CÓDIGO AQUI
subcategorias = ...

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert list(subcategorias.columns) == ["avaliacoes", "nota_media", "pct_nota_1", "diferenca"]
assert (subcategorias["avaliacoes"] >= 800).all() and subcategorias["diferenca"].is_monotonic_decreasing
assert np.allclose(subcategorias["pct_nota_1"] - subcategorias["diferenca"], (df["overall_rating"] == 1).mean())
print(len(subcategorias), "subcategorias")
subcategorias.head(5).round(3)

**3b)** Qual subcategoria tem o pior resultado? Leia dez avaliações de nota 1 dela (use `sample` com `random_state`) e descreva, em duas ou três frases, o que os clientes reclamam.

In [ ]:
# ✏️ Questão 3b
# SEU CÓDIGO AQUI
...

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,0 pontos): clientes

a) Construa `clientes`, com uma linha por `reviewer_id` e as colunas `avaliacoes`, `nota_media`, `primeira` (data da primeira avaliação) e `ultima` (data da última).

b) Junte essas informações à tabela de avaliações, em `com_cliente`, com um `merge` que **confira a cardinalidade** e não altere o número de linhas.

c) Calcule `pct_recorrentes`: a proporção de **clientes** com mais de uma avaliação.

In [ ]:
# ✏️ Questão 4
# SEU CÓDIGO AQUI
clientes = ...
com_cliente = ...
pct_recorrentes = ...

In [ ]:
# ✅ Conferência da questão 4 (não altere)
assert len(clientes) == df["reviewer_id"].nunique() and {"avaliacoes", "nota_media", "primeira", "ultima"} <= set(clientes.columns)
assert len(com_cliente) == len(df), "o merge alterou o número de linhas"
assert clientes["avaliacoes"].sum() == len(df) and (clientes["ultima"] >= clientes["primeira"]).all()
assert 0.05 < pct_recorrentes < 0.20
print(f"{len(clientes)} clientes · {pct_recorrentes:.1%} com mais de uma avaliação")

**4b)** Clientes recorrentes dão notas diferentes dos demais? Calcule a nota média das avaliações de cada grupo e comente. Explique também por que este resultado importa para quem vai separar treino e teste com estes dados.

In [ ]:
# ✏️ Questão 4b
# SEU CÓDIGO AQUI
...

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,5 ponto): evolução semanal

a) Construa `semanal`, indexada pela semana, com as colunas `avaliacoes`, `nota_media` e `pct_negativas` (proporção de notas 1 ou 2).

b) Guarde em `pior_semana` a data (índice) da semana com a maior `pct_negativas`, considerando apenas semanas com pelo menos 3000 avaliações.

In [ ]:
# ✏️ Questão 5
# SEU CÓDIGO AQUI
semanal = ...
pior_semana = ...

In [ ]:
# ✅ Conferência da questão 5 (não altere)
assert {"avaliacoes", "nota_media", "pct_negativas"} <= set(semanal.columns) and 20 <= len(semanal) <= 23
assert semanal["avaliacoes"].sum() == len(df) and semanal.loc[pior_semana, "avaliacoes"] >= 3000
assert semanal.loc[pior_semana, "pct_negativas"] == semanal.query("avaliacoes >= 3000")["pct_negativas"].max()
print("Formato correto. Pior semana com volume:", pior_semana.date())

**5b)** Por que a exigência de um volume mínimo de avaliações ao procurar a pior semana? Olhe o número de avaliações da primeira e da última semana e explique o que acontece com elas.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 6 (1,0 ponto): três achados

Escreva **três achados** sobre estes dados que seriam úteis para quem vai construir um modelo de satisfação do cliente. Cada achado deve: (1) citar um número que você calculou neste notebook ou em uma célula nova; (2) dizer por que ele importa. Declare também como usou assistentes de IA nesta lista.

✏️ **Sua resposta:**

_Escreva aqui._